# Exercise 04 — Learnable Activations (a learned mix of activations)

## Your task

Instead of picking one fixed activation (ReLU *or* GELU *or* SiLU), let the layer **learn
how much each contributes**. With trainable scalars $\alpha_k$ and a fixed dictionary
$\{\mathrm{ReLU}, \mathrm{GELU}, \mathrm{SiLU}\}$:

$$\varphi_{\mathrm{LA}}(z) = \alpha_{\mathrm{relu}}\,\mathrm{ReLU}(z)
+ \alpha_{\mathrm{gelu}}\,\mathrm{GELU}(z)
+ \alpha_{\mathrm{silu}}\,\mathrm{SiLU}(z)$$

The coefficients are input-independent (the same $\alpha_k$ for every sample and feature)
and are trained by gradient descent like any other parameter.

As in the other exercises you implement **only `forward`**. The combination is built from
ops the engine already differentiates (`*`, `+`, `relu`, `gelu`, `exp`, ...), so the engine
produces the backward pass for you. The gradient check (Part 3) confirms it.

## Relation to the other exercises

- [Exercise 01](q01_activations.ipynb): there you built `swish` (= SiLU) as a *primitive*
  op with a hand-written `_backward`. Here SiLU is instead **composed** from engine ops
  (given as `silu` below), so this notebook stands alone — same maths, no `_backward` to
  write.
- [Exercise 02](q02_rewrite_the_stars.ipynb) / [Exercise 03](q03_gated_linear_units.ipynb):
  those *gate* or *multiply* branches; this one **adds** activations with learned weights.
  All three share the same lesson — compose Tensor ops and autograd does the rest — and the
  same `gradient_check` harness.

## The maths of the backward (Part 2 — for understanding; you do NOT code it)

For $y = \sum_k \alpha_k \varphi_k(x)$ the engine derives, automatically:

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \odot
\sum_k \alpha_k \varphi_k'(x) \qquad \text{(chain rule through each } \varphi_k)$$

$$\frac{\partial L}{\partial \alpha_k} = \sum \left(\frac{\partial L}{\partial y} \odot
\varphi_k(x)\right) \qquad \text{(summed over all batch/feature dims)}$$

The $\alpha_k$ gradient is a *sum* because one scalar multiplies the whole tensor — the
engine's `*` backward calls `_unbroadcast`, which sums a `(1, 1)` parameter's gradient over
every position. You get both for free.

## Required reading

[1] Wang, M., Wang, J., Xia, Y., Shen, K., & Zhong, S. *More Expressive Feedforward Layers:
Part I. Token-Adaptive Mixing of Activations.* 2026.

## Conventions

Column-oriented like the rest of the library: tensors are `(features, batch)` and the
coefficients are `(1, 1)` so they broadcast over both axes.

## How to work

1. Run the setup cell.
2. Fill in the two `forward` methods (remove each `raise NotImplementedError`).
3. Run the **grading** cell, then the **mixing** cell to see how much each activation
   actually contributes.

> **Tip:** a `(1,1)` `Parameter` multiplied by a tensor broadcasts and is differentiated by
> the engine, so each weighted term is a one-liner and the layer is their sum. See
> `StarLinear` in Exercise 02 for the shape.

In [ ]:
# Run me first: make ``bert_cpu`` importable whether Jupyter was started in the
# project root or inside ``exercises/``, exactly like the scripts in this folder do.
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "bert_cpu").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np

from bert_cpu import engine as cpu
from bert_cpu import nn
from exercises.grading import gradient_check, report

print("ready — engine imported from", ROOT)

## GIVEN — SiLU, composed from engine ops

The `swish` you hand-wrote in Exercise 01, this time built out of existing ops so this
notebook stands alone.

In [ ]:
def _sigmoid(t: cpu.Tensor) -> cpu.Tensor:
    return 1.0 / (1.0 + (-t).exp())


def silu(t: cpu.Tensor) -> cpu.Tensor:
    """SiLU(t) = t * sigmoid(t)."""
    return t * _sigmoid(t)

## Part 1 — the unconstrained Learnable Activation

A learned linear combination
$\alpha_{\mathrm{relu}}\,\mathrm{ReLU} + \alpha_{\mathrm{gelu}}\,\mathrm{GELU}
+ \alpha_{\mathrm{silu}}\,\mathrm{SiLU}$.

The three coefficients are `(1, 1)` `Parameter`s, initialised to 1 (so the layer starts as
the plain *sum* of the activations, per the paper). They are collected automatically by
`Module.parameters()` and trained like weights.

In [ ]:
class LearnableActivation(nn.Module):
    """A learned linear combination of ReLU, GELU and SiLU."""

    def __init__(self) -> None:
        self.alpha_relu = nn.Parameter(cpu.ones(1, 1))
        self.alpha_gelu = nn.Parameter(cpu.ones(1, 1))
        self.alpha_silu = nn.Parameter(cpu.ones(1, 1))

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # TODO: 
        raise NotImplementedError("implement LearnableActivation.forward")

    def coefficients(self) -> dict:
        """The current (unconstrained) coefficients, for reporting."""
        return {
            "alpha_relu": self.alpha_relu.data.item(),
            "alpha_gelu": self.alpha_gelu.data.item(),
            "alpha_silu": self.alpha_silu.data.item(),
        }

## Part 6 — the softmax-normalized variant

Same idea, but the weights are a softmax over trainable logits $\beta_k$:

$$\pi_k = \frac{e^{\beta_k}}{\sum_j e^{\beta_j}}
\qquad \text{so } \pi_k > 0 \text{ and } \sum_k \pi_k = 1$$

$$\varphi(x) = \sum_k \pi_k \varphi_k(x)$$

You train the *logits* $\beta_k$ (initialised to 0 → equal $\pi = 1/3$), not the $\pi$
directly. This keeps the mix a convex combination: bounded, interpretable, and stable, at
the cost of forbidding negative weights.

In [ ]:
class NormalizedLearnableActivation(nn.Module):
    """Same idea, but the weights are a softmax over trainable logits."""

    def __init__(self) -> None:
        self.beta_relu = nn.Parameter(cpu.zeros(1, 1))
        self.beta_gelu = nn.Parameter(cpu.zeros(1, 1))
        self.beta_silu = nn.Parameter(cpu.zeros(1, 1))

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # TODO: 
        raise NotImplementedError("implement NormalizedLearnableActivation.forward")

    def coefficients(self) -> dict:
        """The normalized weights pi_k, for reporting."""
        betas = np.array([self.beta_relu.data, self.beta_gelu.data, self.beta_silu.data]).ravel()
        e = np.exp(betas - betas.max())
        pi = e / e.sum()
        return {"pi_relu": float(pi[0]), "pi_gelu": float(pi[1]), "pi_silu": float(pi[2])}

## GIVEN — Part 3, the gradient check

You do not edit the cells below.

In [ ]:
def grade() -> None:
    """Part 3: gradient-check both layers (analytic backward vs finite diff)."""
    cpu.set_seed(0)
    x = cpu.Tensor(np.random.randn(4, 3))
    target = cpu.Tensor(np.random.randn(4, 3))

    print("Gradient check (analytic backward vs finite differences):\n")
    all_ok = True
    for name, layer in [
        ("LearnableActivation", LearnableActivation()),
        ("NormalizedLearnableActivation", NormalizedLearnableActivation()),
    ]:
        print(f"  {name}:")
        try:
            worst = gradient_check(layer, x, target)
        except NotImplementedError as exc:
            all_ok = False
            print(f"    -> SKIPPED ({exc}).\n")
            continue
        ok = worst < 1e-5
        all_ok = all_ok and ok
        print(f"    -> {'PASS' if ok else 'FAIL'} (worst {worst:.2e})\n")
    report(all_ok, "layers")


grade()

## A coefficient is not an importance

Forward-only: with all coefficients equal to 1, each activation still contributes a
*different* amount — so a coefficient's size alone does not say how important an activation
is. You must weigh $\alpha_k$ against the activation's output scale.

In [ ]:
def demonstrate_mix() -> None:
    """With every coefficient = 1, how much does each term actually contribute?"""
    cpu.set_seed(0)
    la = LearnableActivation()                       # coefficients all 1.0
    # A negative-shifted input: ReLU zeros most of it, GELU/SiLU do not — so the
    # same coefficient buys very different contributions.
    x = cpu.Tensor(np.random.randn(1, 256) - 1.5)
    terms = {
        "alpha_relu * ReLU(x)": (la.alpha_relu * x.relu()).data,
        "alpha_gelu * GELU(x)": (la.alpha_gelu * x.gelu()).data,
        "alpha_silu * SiLU(x)": (la.alpha_silu * silu(x)).data,
    }
    print("With every coefficient = 1, mean |contribution| of each term:")
    for name, value in terms.items():
        print(f"    {name:22s} {float(np.abs(value).mean()):.3f}")
    print("Equal coefficients do NOT mean equal contribution — the activations"
          "\ndiffer in output scale, so read each alpha_k together with phi_k(x).")


demonstrate_mix()

---

**That is the last fill-in exercise.** From here the folder has two *capstones* — complete,
runnable training baselines rather than blanks to fill:

```bash
python -m exercises.task_binary_classification   # UCI Adult, Adam, cross-entropy
python -m exercises.task_learn_embedding         # word2vec on the Flatland corpus
```